In [1]:
!nvidia-smi
!nvcc --version

Fri Nov  6 20:47:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile hello.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void helloKernel()
{
    printf("Hello from the GPU!\n");
}

int main()
{
    printf("Hello from the CPU!\n");

    helloKernel<<<1, 1>>>();

    cudaDeviceSynchronize();

    return 0;
}

Writing hello.cu


In [3]:
!nvcc hello.cu -o hello
!./hello

Hello from the CPU!
Hello from the GPU!


In [4]:
%%writefile blocks_threads.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void helloKernel()
{
    printf("Block %d, Thread %d\n",
           blockIdx.x,
           threadIdx.x);
}

int main()
{
    printf("Hello from the CPU!\n");

    helloKernel<<<2, 4>>>();

    cudaDeviceSynchronize();

    return 0;
}

Writing blocks_threads.cu


In [5]:
!nvcc blocks_threads.cu -o blocks_threads
!./blocks_threads

Hello from the CPU!
Block 1, Thread 0
Block 1, Thread 1
Block 1, Thread 2
Block 1, Thread 3
Block 0, Thread 0
Block 0, Thread 1
Block 0, Thread 2
Block 0, Thread 3


In [6]:
%%writefile global_index.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void showIndex()
{
    int index =
        threadIdx.x +
        blockIdx.x * blockDim.x;

    printf("Block %d, Thread %d, Global index %d\n",
           blockIdx.x,
           threadIdx.x,
           index);
}

int main()
{
    showIndex<<<3, 4>>>();

    cudaDeviceSynchronize();

    return 0;
}

Writing global_index.cu


In [7]:
!nvcc global_index.cu -o global_index
!./global_index

Block 2, Thread 0, Global index 8
Block 2, Thread 1, Global index 9
Block 2, Thread 2, Global index 10
Block 2, Thread 3, Global index 11
Block 1, Thread 0, Global index 4
Block 1, Thread 1, Global index 5
Block 1, Thread 2, Global index 6
Block 1, Thread 3, Global index 7
Block 0, Thread 0, Global index 0
Block 0, Thread 1, Global index 1
Block 0, Thread 2, Global index 2
Block 0, Thread 3, Global index 3


In [8]:
%%writefile vector_add.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(const float *A,
                          const float *B,
                          float *C,
                          int N)
{
    int i =
        threadIdx.x +
        blockIdx.x * blockDim.x;

    if (i < N)
    {
        C[i] = A[i] + B[i];
    }
}

int main()
{
    const int N = 1000000;

    size_t bytes = N * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C = (float *)malloc(bytes);

    for (int i = 0; i < N; ++i)
    {
        A[i] = (float)i;
        B[i] = 2.0f * i;
    }

    float *dev_A;
    float *dev_B;
    float *dev_C;

    // allocate GPU memory
    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    // copy A and B to GPU memory
    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;

    int blocks =
        (N + threadsPerBlock - 1)
        / threadsPerBlock;

    // launch vectorAdd kernel
    vectorAdd<<<blocks, threadsPerBlock>>>(dev_A, dev_B, dev_C, N);

    // Wait until the GPU finishes
    cudaDeviceSynchronize();

    // copy C back to the CPU
    cudaMemcpy(C, dev_C, bytes, cudaMemcpyDeviceToHost);

    printf("C[0] = %f\n", C[0]);
    printf("C[10] = %f\n", C[10]);
    printf("C[N-1] = %f\n", C[N - 1]);

    // check every element: C[i] must be 3i
    int errors = 0;
    for (int i = 0; i < N; ++i)
    {
        if (C[i] != 3.0f * i)
        {
            errors++;
        }
    }
    printf("Errors: %d\n", errors);

    // free device memory
    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C);

    return 0;
}

Writing vector_add.cu


In [9]:
!nvcc vector_add.cu -o vector_add
!./vector_add

C[0] = 0.000000
C[10] = 30.000000
C[N-1] = 2999997.000000
Errors: 0


In [10]:
%%writefile vector_add_blocks.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(const float *A,
                          const float *B,
                          float *C,
                          int N)
{
    int i =
        threadIdx.x +
        blockIdx.x * blockDim.x;

    if (i < N)
    {
        C[i] = A[i] + B[i];
    }
}

int main(int argc, char **argv)
{
    const int N = 1000000;

    size_t bytes = N * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C = (float *)malloc(bytes);

    for (int i = 0; i < N; ++i)
    {
        A[i] = (float)i;
        B[i] = 2.0f * i;
    }

    float *dev_A;
    float *dev_B;
    float *dev_C;

    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);

    // threads per block comes from the command line, default is 256
    int threadsPerBlock = 256;
    if (argc > 1)
    {
        threadsPerBlock = atoi(argv[1]);
    }

    int blocks =
        (N + threadsPerBlock - 1)
        / threadsPerBlock;

    vectorAdd<<<blocks, threadsPerBlock>>>(dev_A, dev_B, dev_C, N);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, bytes, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int i = 0; i < N; ++i)
    {
        if (C[i] != 3.0f * i)
        {
            errors++;
        }
    }

    printf("Threads per block: %d, Blocks: %d\n", threadsPerBlock, blocks);
    printf("C[0] = %f, C[10] = %f, C[N-1] = %f, Errors: %d\n\n",
           C[0], C[10], C[N - 1], errors);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C);

    return 0;
}

Writing vector_add_blocks.cu


In [11]:
!nvcc vector_add_blocks.cu -o vector_add_blocks
!for t in 32 64 128 256 512; do ./vector_add_blocks $t; done

Threads per block: 32, Blocks: 31250
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 64, Blocks: 15625
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 128, Blocks: 7813
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 256, Blocks: 3907
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 512, Blocks: 1954
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0



In [12]:
%%writefile vector_add_timing.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(const float *A,
                          const float *B,
                          float *C,
                          int N)
{
    int i =
        threadIdx.x +
        blockIdx.x * blockDim.x;

    if (i < N)
    {
        C[i] = A[i] + B[i];
    }
}

int main(int argc, char **argv)
{
    const int N = 1000000;

    size_t bytes = N * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C = (float *)malloc(bytes);

    for (int i = 0; i < N; ++i)
    {
        A[i] = (float)i;
        B[i] = 2.0f * i;
    }

    float *dev_A;
    float *dev_B;
    float *dev_C;

    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    if (argc > 1)
    {
        threadsPerBlock = atoi(argv[1]);
    }

    int blocks =
        (N + threadsPerBlock - 1)
        / threadsPerBlock;

    // warm up launch, not measured (the first launch is always slower)
    vectorAdd<<<blocks, threadsPerBlock>>>(dev_A, dev_B, dev_C, N);
    cudaDeviceSynchronize();

    cudaEvent_t start, stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    const int RUNS = 10;
    float total = 0.0f;

    printf("Threads per block: %d\n", threadsPerBlock);
    printf("Times (ms):");

    for (int r = 0; r < RUNS; ++r)
    {
        cudaEventRecord(start);

        vectorAdd<<<blocks, threadsPerBlock>>>(
            dev_A, dev_B, dev_C, N);

        cudaEventRecord(stop);

        cudaEventSynchronize(stop);

        float milliseconds = 0.0f;

        cudaEventElapsedTime(
            &milliseconds, start, stop);

        printf(" %.4f", milliseconds);
        total += milliseconds;
    }

    printf("\nAverage kernel time: %f ms\n", total / RUNS);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaMemcpy(C, dev_C, bytes, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int i = 0; i < N; ++i)
    {
        if (C[i] != 3.0f * i)
        {
            errors++;
        }
    }

    printf("C[0] = %f, C[10] = %f, C[N-1] = %f, Errors: %d\n\n",
           C[0], C[10], C[N - 1], errors);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C);

    return 0;
}

Writing vector_add_timing.cu


In [13]:
!nvcc vector_add_timing.cu -o vector_add_timing
!for t in 32 64 128 256 512; do ./vector_add_timing $t; done

Threads per block: 32
Times (ms): 0.1106 0.1087 0.1054 0.1086 0.1065 0.1089 0.1050 0.1092 0.1066 0.1092
Average kernel time: 0.107869 ms
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 64
Times (ms): 0.0553 0.0519 0.0513 0.0506 0.0512 0.0513 0.0512 0.0508 0.0512 0.0509
Average kernel time: 0.051578 ms
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 128
Times (ms): 0.0554 0.0532 0.0513 0.0497 0.0512 0.0493 0.0501 0.0509 0.0512 0.0512
Average kernel time: 0.051347 ms
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 256
Times (ms): 0.0571 0.0519 0.0512 0.0508 0.0509 0.0512 0.0516 0.0501 0.0514 0.0500
Average kernel time: 0.051619 ms
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 512
Times (ms): 0.0553 0.0532 0.0532 0.0506 0.0501 0.0514 0.0516 0.0499 0.0513 0.0529
Average kernel time: 0.051952 ms
C[0] = 0.000000, C[10] =